# Dataset Build Colab

This notebook supports exactly one Dataset Build execution path:

1. Mount Google Drive at `/content/drive`.
2. Read translations only from:
   - `/content/drive/MyDrive/text-to-sign-production/raw/how2sign/translations/how2sign_realigned_train.csv`
   - `/content/drive/MyDrive/text-to-sign-production/raw/how2sign/translations/how2sign_realigned_val.csv`
   - `/content/drive/MyDrive/text-to-sign-production/raw/how2sign/translations/how2sign_realigned_test.csv`
3. Read keypoints only from `.tar.zst` archives at:
   - `/content/drive/MyDrive/text-to-sign-production/raw/how2sign/archives/train_2D_keypoints.tar.zst`
   - `/content/drive/MyDrive/text-to-sign-production/raw/how2sign/archives/val_2D_keypoints.tar.zst`
   - `/content/drive/MyDrive/text-to-sign-production/raw/how2sign/archives/test_2D_keypoints.tar.zst`
4. Copy archives into `/content/how2sign_downloads`, extract there, and stage the canonical repo raw layout.
5. Run Dataset Build through the shared workflow orchestrator.
6. Package outputs and copy the archives only to `/content/drive/MyDrive/text-to-sign-production/artifacts/dataset-build/processed-v1/`.

The notebook is orchestration-only. Dataset logic stays in repository modules and this notebook calls the stage workflow directly so progress bars can render live in Colab.


## 1. Environment And Repository Setup

This section installs the minimal Colab prerequisites, clones the repository, installs Python dependencies, and prepares direct imports from `src/`.


In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

print(f"Python: {sys.version.split()[0]}")

if shutil.which("zstd") is None:
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "zstd"], check=True)
else:
    print("zstd is already available.")

REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "master"
WORKTREE_ROOT = Path("/content/text-to-sign-production")
RUNTIME_ROOT = WORKTREE_ROOT.parent

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
try:
    current_cwd = Path.cwd()
except FileNotFoundError:
    current_cwd = None

if current_cwd is None or current_cwd == WORKTREE_ROOT or WORKTREE_ROOT in current_cwd.parents:
    os.chdir(RUNTIME_ROOT)

if WORKTREE_ROOT.exists():
    shutil.rmtree(WORKTREE_ROOT)

subprocess.run(["git", "clone", REPO_URL, str(WORKTREE_ROOT)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=WORKTREE_ROOT, check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip"],
    cwd=WORKTREE_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"],
    cwd=WORKTREE_ROOT,
    check=True,
)
os.chdir(WORKTREE_ROOT)

if str(WORKTREE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(WORKTREE_ROOT / "src"))

current_revision = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=WORKTREE_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Repository ready at {WORKTREE_ROOT}")
print(f"Checked out revision: {current_revision}")
print("Python import path ready for direct library calls.")

## 2. Mount Google Drive

Edit only `PIPELINE_SPLITS` if you want a subset such as `["val"]` or `["test"]`. All input and output paths are fixed by the repository.


In [ ]:
from google.colab import drive

PIPELINE_SPLITS = ["train", "val", "test"]

drive.mount("/content/drive", force_remount=False)
print(f"Selected splits: {PIPELINE_SPLITS}")
print("Fixed input root: /content/drive/MyDrive/text-to-sign-production/raw/how2sign")
print("Fixed local archive workspace: /content/how2sign_downloads")
print(
    "Fixed output root: "
    "/content/drive/MyDrive/text-to-sign-production/artifacts/dataset-build/processed-v1/"
)

## 3. Run Dataset Build

This calls the shared stage workflow directly in the notebook kernel. The workflow stages fixed Drive inputs, builds manifests and samples, packages outputs, and publishes archives to the fixed Drive artifact path.


In [ ]:
from text_to_sign_production.workflows.dataset_build import run_dataset_build

result = run_dataset_build(
    splits=tuple(PIPELINE_SPLITS),
    filter_config_path=WORKTREE_ROOT / "configs/data/filter-v1.yaml",
    input_mode="fixed_colab_drive",
    output_mode="fixed_colab_drive",
)

print(f"Dataset Build completed for splits: {', '.join(result.splits)}")
for item in result.staged_inputs:
    print(
        f"staged {item['split']}: translation={item['translation_path']} "
        f"keypoints={item['keypoints_path']}"
    )
for split in result.splits:
    raw_report = result.assumption_report["splits"][split]
    filter_report = result.filter_report["splits"][split]
    print(
        f"{split}: rows={raw_report['translation_row_count']} "
        f"matched={raw_report['matched_sample_count']} "
        f"kept={filter_report['kept_samples']} "
        f"dropped={filter_report['dropped_samples']}"
    )
for path in result.output_paths:
    print(f"published: {path}")